# DeMoSearch — test the complete inference pipeline

Use **existing model checkpoints** to test wall analysis. There are no dataset-preparation, YOLO training or model-training validation steps in this notebook.

| Mode | Input | Pipeline | Result |
| --- | --- | --- | --- |
| 2D | One photo | SAM3 masks → RGB hold types → colors/positions | `scene.json` |
| 3D | Overlapping photos/ZIP or a video | COLMAP wall reconstruction → SAM3 masks → RGB types → multiview mapping | Same JSON schema plus a rotatable GLB with matching evidence |

All implementation lives in Python modules. This ordinary `.ipynb` is edited directly and only calls those modules through the CLI. The graph algorithm receives the final wall/hold JSON and creates the route separately.

Run the setup and configure input/checkpoint paths. Enable `RUN_YOLO_PREVIEW` to inspect one image with your checkpoint or `RUN_PIPELINE` to export a complete scene. A CUDA runtime is required for SAM3; 3D additionally needs a CUDA-enabled COLMAP executable.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

INSTALL_SAM3 = True  # Set False for a YOLO-only preview in Colab.
ON_COLAB = Path("/content").is_dir()
WORK = Path("/content/demosearch-inference") if ON_COLAB else Path.cwd()
if ON_COLAB:
    PROJECT = WORK / "project"
    WORK.mkdir(parents=True, exist_ok=True)
    if not PROJECT.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/DiasKzStrong/hold_detection.git", str(PROJECT),
        ], check=True)
    else:
        subprocess.run(["git", "pull", "--ff-only"], cwd=PROJECT, check=True)
else:
    PROJECT = Path.cwd().resolve()
    if not (PROJECT / "src" / "hold_detection").is_dir():
        PROJECT = PROJECT.parent

if not (PROJECT / "src" / "hold_detection").is_dir():
    raise FileNotFoundError("Set PROJECT to the hold_detection repository checkout.")

PYTHON = Path(sys.executable)
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))

if ON_COLAB:
    from importlib import reload
    from hold_detection import notebook_runtime

    PYTHON = reload(notebook_runtime).prepare_runtime(WORK, PROJECT, sam3=INSTALL_SAM3)

def run_holds(*arguments, quiet=False):
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    command = [str(PYTHON), "-m", "hold_detection.cli", *map(str, arguments)]
    result = subprocess.run(command, cwd=PROJECT, env=environment, stderr=subprocess.PIPE,
                            stdout=subprocess.DEVNULL if quiet else None, text=True)
    if result.returncode:
        detail = result.stderr.strip()
        raise RuntimeError(f"{arguments[0]} failed (exit {result.returncode}). {detail}")
    if result.stderr:
        print(result.stderr, file=sys.stderr, end="")

print("Project:", PROJECT)
print("Inference Python:", PYTHON)

## Input and ready checkpoints

Set `MODE="2d"` for one photograph; choose `"3d"` for a directory/ZIP of overlapping photos or a video. `INPUT_PATH_2D` points to the Blue V5 photo. `INPUT_PATH_3D` points to `wallC_raw_data.zip` in the same Drive folder. The preset stages **15 overlapping views of the Blue V5 section** from `keyframes/corrected_images`. Set `INPUT_PATH_3D` to your own photo folder, ZIP, or video of a wall section to use a different capture. Set local paths when running outside Colab.

`TYPE_WEIGHTS` must point to an **already fine-tuned six-class hold detector**, not the generic COCO `yolo26m.pt`. If you deliberately disable `USE_TYPE_MODEL`, SAM3 can still run but grip types require review.

Set `TYPE_IMGSZ=640` for the 640px Roboflow checkpoint. The same setting is used by the YOLO preview and SAM3 type matching.

The optional checkpoint-download cell fetches official SAM3 weights only. It does not train a model.

In [ ]:
MOUNT_DRIVE = ON_COLAB
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

MODE = "2d"  # "2d" or "3d"
WALL_C = Path("/content/drive/MyDrive/CV26_FinalProj/DeMoSearch-data/wallC_overhang_right")
WALL_C_ARCHIVE = WALL_C / "wallC_raw_data.zip"
INPUT_PATH_2D = WALL_C / "blue_v5" / "blue_v5.HEIC"
INPUT_PATH_3D = WALL_C_ARCHIVE
INPUT_PATH = INPUT_PATH_3D if MODE == "3d" else INPUT_PATH_2D
SAM_WEIGHTS = PROJECT / "models" / "sam3" / "sam3.pt"
TYPE_WEIGHTS = Path("/content/drive/MyDrive/CV26_FinalProj/Models/yolo_hold_cls.pt") if ON_COLAB else PROJECT / "Models" / "yolo_hold_cls.pt"
TYPE_IMGSZ = 640
USE_TYPE_MODEL = True
RUN_YOLO_PREVIEW = False
FETCH_SAM3_CHECKPOINT = True
RUN_PIPELINE = True

RUN_NAME = "photo_demo" if MODE == "2d" else "wallc_section_3d"
OUTPUT = WORK / "results" / RUN_NAME if ON_COLAB else PROJECT / "runs" / RUN_NAME
COLMAP = "colmap"
MAX_FRAMES = 48
CONFIDENCE = 0.5
SCALE_M_PER_UNIT = None
SCALE_REFERENCE = None
CORRECTIONS = None  # Optional JSON keyed by hold ID; apply after inspecting the first result.

print("Mode:", MODE, "| input:", INPUT_PATH)
print("Output:", OUTPUT)

In [ ]:
if FETCH_SAM3_CHECKPOINT and not SAM_WEIGHTS.is_file():
    import getpass
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass.getpass("Approved Hugging Face read token (hidden): ")
    run_holds("fetch-sam3", "--output", SAM_WEIGHTS.parent)

print("SAM3 checkpoint exists:", SAM_WEIGHTS.is_file())
print("Hold-type checkpoint exists:", TYPE_WEIGHTS.is_file())
if RUN_PIPELINE or RUN_YOLO_PREVIEW:
    if MODE not in {"2d", "3d"}:
        raise ValueError("MODE must be 2d or 3d.")
    if not INPUT_PATH.exists():
        raise FileNotFoundError(INPUT_PATH)
if RUN_PIPELINE and not SAM_WEIGHTS.is_file():
    raise FileNotFoundError("Set SAM_WEIGHTS or enable FETCH_SAM3_CHECKPOINT.")
if (RUN_YOLO_PREVIEW or (RUN_PIPELINE and USE_TYPE_MODEL)) and not TYPE_WEIGHTS.is_file():
    raise FileNotFoundError("Set TYPE_WEIGHTS to your exported hold-type .pt checkpoint.")

## 1. Prepare and preview the capture

Images are oriented and bounded to 1600px. Video is sampled into up to `MAX_FRAMES` views. The 2D mode accepts exactly one photo, including HEIC/HEIF. For 3D, use a sharp capture with parallax and substantial overlap.

Completed preparation can be reused when rerunning the same input/settings. Choose a new output directory after changing input or model settings.

In [ ]:
# The Wall C demo is limited to overlapping views of the Blue V5 section.
WALL_C_FRAME_IDS = (
    "19943171389", "19947467812", "19951347911", "19953246327", "19955644324",
    "19957559391", "19959024833", "19961472786", "19963237977", "19964237141",
    "19992363549", "20002987951", "20006168609", "20009232698", "20011997039",
)
CAPTURE = OUTPUT / "capture"
if RUN_PIPELINE or RUN_YOLO_PREVIEW:
    capture_input = INPUT_PATH
    if MODE == "3d" and INPUT_PATH == WALL_C_ARCHIVE:
        from hashlib import sha256
        from zipfile import ZipFile
        from PIL import Image

        stats = INPUT_PATH.stat()
        preset_id = (str(INPUT_PATH), stats.st_size, stats.st_mtime_ns, WALL_C_FRAME_IDS)
        preset_key = sha256(repr(preset_id).encode()).hexdigest()[:12]
        capture_input = OUTPUT / "wallc_inputs" / preset_key
        capture_input.mkdir(parents=True, exist_ok=True)
        with ZipFile(INPUT_PATH) as archive:
            for frame_id in WALL_C_FRAME_IDS:
                target = capture_input / f"{frame_id}.png"
                if not target.is_file():
                    member = f"keyframes/corrected_images/{frame_id}.jpg"
                    temporary = target.with_suffix(".tmp")
                    with archive.open(member) as source, Image.open(source) as image:
                        upright = image.convert("RGB").transpose(Image.Transpose.ROTATE_270)
                        upright.save(temporary, format="PNG")
                    temporary.replace(target)
        print("Wall C section input folder", capture_input)
    run_holds("prepare-capture", capture_input, "--mode", MODE, "--output", CAPTURE,
              "--max-frames", MAX_FRAMES)

capture_path = CAPTURE / "capture.json"
if capture_path.is_file():
    manifest = json.loads(capture_path.read_text())
    print("Prepared frames:", len(manifest["frames"]))
    from IPython.display import display
    from PIL import Image
    for frame in manifest["frames"][:3]:
        with Image.open(CAPTURE / frame["path"]) as image:
            preview = image.copy()
            preview.thumbnail((600, 450))
            display(preview)
else:
    print("Set RUN_PIPELINE=True or RUN_YOLO_PREVIEW=True to prepare the selected input.")

### Optional checkpoint preview

Set `RUN_YOLO_PREVIEW=True` to inspect the YOLO boxes, types and scores on the first prepared image. The preview can run on CPU and does not require a SAM3 checkpoint. Use the full pipeline for instance masks, color groups and the downstream scene.

In [ ]:
if RUN_YOLO_PREVIEW:
    from uuid import uuid4

    first_frame = CAPTURE / manifest["frames"][0]["path"]
    preview_run = OUTPUT / f"yolo_preview_{TYPE_IMGSZ}_{uuid4().hex[:8]}"
    run_holds("predict", first_frame, "--output", preview_run, "--weights", TYPE_WEIGHTS,
              "--classifier", "detector", "--imgsz", TYPE_IMGSZ, "--conf", 0.2,
              "--min-type-score", 0.5, "--device", "auto")
    summary = json.loads((preview_run / "summary.json").read_text())
    frame = summary["frames"][0]
    print("YOLO proposals", frame["hold_count"], "| types", frame["objects"])
    with Image.open(preview_run / frame["frame_id"] / "annotated.jpg") as image:
        image.thumbnail((900, 900))
        display(image)
    print("YOLO review gallery", preview_run / "index.html")

## 2. Reconstruct the wall section — 3D only

COLMAP estimates cameras, stereo depth and a mesh. The pipeline exports GLB with trimesh. SAM3 will use the **undistorted registered images**, so its masks agree with the camera/depth coordinates.

COLMAP must already be installed with CUDA support inside the selected environment/container. Installing `pycolmap` alone does not provide this CLI workflow.

Without calibration, reconstruction coordinates have arbitrary units and meter-valued JSON fields stay `null`. Set a known `SCALE_M_PER_UNIT`, or supply a scale-reference JSON containing two pixels in an **undistorted** image and their known distance. The scene export estimates up from upright camera poses. The calibration section offers a manual override.

In [ ]:
RECONSTRUCTION = OUTPUT / "reconstruction" / "reconstruction.json" if MODE == "3d" else None
if RUN_PIPELINE and MODE == "3d":
    arguments = ["reconstruct-wall", CAPTURE, "--output", OUTPUT / "reconstruction", "--colmap", COLMAP]
    if SCALE_M_PER_UNIT is not None:
        arguments += ["--scale-m-per-unit", SCALE_M_PER_UNIT]
    if SCALE_REFERENCE is not None:
        arguments += ["--scale-reference", SCALE_REFERENCE]
    run_holds(*arguments)

if RECONSTRUCTION is not None and RECONSTRUCTION.is_file():
    reconstruction = json.loads(RECONSTRUCTION.read_text())
    print("Registered images:", reconstruction["registered_images"], "/", reconstruction["capture_images"])
    print("Components:", reconstruction["component_count"], "| units:", reconstruction["units"])
    print("GLB:", reconstruction["glb_path"])
elif MODE == "2d":
    print("2D mode: reconstruction is skipped.")

## 3. Segment holds and infer RGB types

SAM3 provides the instance masks. The ready hold-type detector supplies six-class RGB proposals through one-to-one overlap matching. No fitting or weight updates occur here.

Each frame runs in a fresh SAM3 process. Completed frame batches are stored and reused on resume. Inspect a failed frame's log under `OUTPUT/logs` if a GPU process exits.

In [ ]:
if RUN_PIPELINE:
    arguments = ["segment-capture", CAPTURE, "--run", OUTPUT,
                 "--sam-weights", SAM_WEIGHTS, "--device", "0", "--confidence", CONFIDENCE]
    if USE_TYPE_MODEL:
        arguments += ["--type-weights", TYPE_WEIGHTS, "--type-imgsz", TYPE_IMGSZ]
    if RECONSTRUCTION is not None:
        arguments += ["--reconstruction-path", RECONSTRUCTION]
    run_holds(*arguments)
else:
    print("Inference disabled until RUN_PIPELINE=True.")

## 4. Fuse observations and export the common scene

2D exports frame-local positions/sizes. 3D lifts mask pixels through stereo depth, combines observations of the same Physical Hold and attaches reconstructed-mesh evidence. The export estimates vertical from camera poses and calculates relative hold heights when the estimate passes its quality checks. Single-view and unmapped candidates remain available for review.

Both modes export the same hold-field schema. Unknown types need manual confirmation or an explicit policy in the teammate's algorithm. A partial scan cannot measure hidden grip area, lip depth or true hold volume; unavailable values remain null.

In [ ]:
def export_wall_result(run, mode, reconstruction_path, corrections, up_reference=None):
    arguments = ["export-wall", run, "--mode", mode]
    if reconstruction_path is not None:
        arguments += ["--reconstruction-path", reconstruction_path]
    if corrections is not None:
        arguments += ["--annotations-path", corrections]
    if up_reference is not None:
        arguments += ["--up-reference", up_reference]
    run_holds(*arguments, quiet=True)
    return json.loads((run / "scene.json").read_text())

SCENE_PATH = OUTPUT / "scene.json"
if RUN_PIPELINE:
    saved_reference = OUTPUT / "up_reference.json"
    up_reference = saved_reference if MODE == "3d" and saved_reference.is_file() else None
    export_wall_result(OUTPUT, MODE, RECONSTRUCTION, CORRECTIONS, up_reference)

if SCENE_PATH.is_file():
    scene = json.loads(SCENE_PATH.read_text())
    print("Mode:", scene["analysis_mode"], "| holds:", len(scene["holds"]))
    print("Units:", scene["wall"]["units"])
    print("Review:", scene["review"])
    print("Input for the graph algorithm:", SCENE_PATH)
    print("Review gallery:", OUTPUT / "inference" / "index.html")
    from IPython.display import display
    import pandas as pd
    hold_fields = ["id", "hold_type", "hold_type_confidence", "color_group_id", "observation_count", "geometry_status", "height_3d", "height_m"]
    display(pd.DataFrame(scene["holds"]).reindex(columns=hold_fields).head(15))
else:
    scene = None
    print("No scene.json yet.")

## 5. Check or calibrate up — 3D only

The automatic estimate uses the registered camera poses and assumes the phone was held upright in roll. If the camera angles are too similar or disagree, use the picker. In Colab select a registered frame, click the bottom and top of a real vertical line, verify the arrow, and press **Apply up direction**. Use **Use automatic estimate** to switch back. The picker updates the existing scene without running COLMAP or SAM3 again.

`height_3d` increases in the direction of `wall.up_vector` and uses the reconstruction origin as its zero. `height_m` is available when the reconstruction has metric scale. Rerun the mesh viewer after applying a new direction.

In [ ]:
if scene is not None and scene["analysis_mode"] == "3d":
    from IPython.display import display

    def describe_up(result):
        orientation = result["wall"].get("up_orientation") or {}
        status = orientation.get("status")
        if status == "calibrated":
            return "Up calibrated from the selected vertical. scene.json is updated."
        if status == "estimated":
            return f"Up estimated from {orientation['camera_count']} registered camera poses."
        reason = orientation.get("reason", "run the current scene exporter")
        return "Manual calibration needed (" + reason.replace("_", " ") + ")."

    print(describe_up(scene))
    if ON_COLAB:
        from tempfile import TemporaryDirectory
        from uuid import uuid4
        from google.colab import output as colab_output
        from IPython.display import Javascript, JSON
        from hold_detection.up_picker import picker_frame, picker_javascript

        def show_up_picker(run, reconstruction_path, corrections):
            reconstruction = json.loads(reconstruction_path.read_text())
            depth = Path(reconstruction["depth_directory"])
            images = Path(reconstruction["image_source"])
            frames = [name for name in reconstruction["registered_image_names"]
                      if (images / name).is_file() and (depth / f"{name}.geometric.bin").is_file()]
            if not frames:
                print("Manual calibration needs a registered photo with geometric stereo depth.")
                return
            saved_reference = run / "up_reference.json"

            def get_frame(name):
                if name not in frames:
                    raise ValueError("Choose a registered frame from the dropdown.")
                return JSON(picker_frame(images / name))

            def update_up(reference):
                global scene
                try:
                    if reference is None:
                        updated = export_wall_result(run, "3d", reconstruction_path, corrections)
                        saved_reference.unlink(missing_ok=True)
                    else:
                        with TemporaryDirectory(dir=run, prefix="up-") as directory:
                            candidate = Path(directory) / "reference.json"
                            candidate.write_text(json.dumps(reference, allow_nan=False))
                            updated = export_wall_result(run, "3d", reconstruction_path, corrections, candidate)
                            candidate.replace(saved_reference)
                except (OSError, ValueError, RuntimeError) as error:
                    return JSON({"message": str(error)})
                if SCENE_PATH == run / "scene.json":
                    scene = updated
                return JSON({"message": describe_up(updated) + " Rerun the mesh viewer to refresh it."})

            prefix = "demosearch.up_" + uuid4().hex + "."
            colab_output.register_callback(prefix + "frame", get_frame)
            colab_output.register_callback(prefix + "apply", update_up)
            display(Javascript(picker_javascript(frames, prefix)))

        show_up_picker(OUTPUT, RECONSTRUCTION, CORRECTIONS)
    else:
        print("To calibrate in this runtime, save an up_reference.json in the run directory "
              "and pass it to holds export-wall --up-reference.")

## 6. Review masks, hold types, and 3D matches

Compare the photo on the left with the saved masks and boxes on the right. Use the color selector to inspect types or color groups. Select one hold to see just its mask and the frames that contributed to it.

In 3D, rotate or zoom the GLB preview and hover over a marker to read its hold ID. The same ID is in the photo selector below. Green markers combine multiple views. Amber markers have single-view evidence. Unmapped holds remain in the photo review. Toggle **Lifted centers to mesh vertices** in the legend to inspect the geometric correspondence. Download the full-resolution GLB of the reconstructed section from the viewer.

In [ ]:
if scene is not None:
    from importlib.util import find_spec

    review_packages = {"pillow_heif": "pillow-heif==1.8.1", "ipywidgets": "ipywidgets>=8,<9"}
    if scene["analysis_mode"] == "3d":
        review_packages.update({
            "plotly": "plotly>=6,<7",
            "trimesh": "trimesh>=4.10,<6",
            "fast_simplification": "fast-simplification>=0.1,<1",
        })
    missing = [package for module, package in review_packages.items() if find_spec(module) is None]
    if missing and ON_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    elif missing:
        raise ImportError("Install the review dependencies in the notebook kernel: " + ", ".join(missing))

if scene is not None and scene["analysis_mode"] == "3d":
    from IPython.display import FileLink, display
    from hold_detection.wall_review import hold_labels, wall_mesh_figure

    glb_path = Path(scene["wall"]["glb_path"])
    figure = wall_mesh_figure(scene, hold_labels(scene))
    if ON_COLAB:
        figure.show(renderer="colab")
        import ipywidgets as widgets
        from google.colab import files

        download = widgets.Button(description="Download wall.glb", icon="download")
        download.on_click(lambda _: files.download(str(glb_path)))
        display(download)
    else:
        figure.show()
        display(FileLink(os.path.relpath(glb_path)))

In [ ]:
if scene is not None:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import display
    from hold_detection.wall_review import hold_labels, render_frame_overlay

    labels = hold_labels(scene)
    holds_by_id = {hold["id"]: hold for hold in scene["holds"]}
    observation_frame = {obs["id"]: obs["frame_id"] for obs in scene["observations"]}
    frames = {}
    for path in sorted((OUTPUT / "inference").glob("*/holds.json")):
        frame = json.loads(path.read_text())
        frames[frame["frame_id"]] = frame
    original_names = {
        Path(item["path"]).name: Path(item["source"]).name for item in manifest["frames"]
    } if capture_path.is_file() else {}
    frame_options = []
    for frame_id, frame in frames.items():
        name = Path(frame["source_image"]).name
        source_name = original_names.get(name, name)
        frame_options.append((f"{source_name} · {len(frame['holds'])} observations", frame_id))

    if frame_options:
        review_3d = scene["analysis_mode"] == "3d"
        hold_options = [("All holds", None)] + [
            (f"{labels[h['id']]} · {h['hold_type']} · {h['observation_count']} view(s)", h["id"])
            for h in scene["holds"]
        ]
        color_options = [("Hold type", "type"), ("Color group", "group")]
        if review_3d:
            color_options.insert(0, ("3D matching status", "geometry"))
        hold_menu = widgets.Dropdown(options=hold_options, description="Hold")
        frame_menu = widgets.Dropdown(options=frame_options, description="Frame")
        color_menu = widgets.Dropdown(options=color_options, description="Color by")
        review_output = widgets.Output()

        def show_frame(change=None):
            frame_id = frame_menu.value
            if frame_id is None:
                return
            hold_id = hold_menu.value
            original, overlay, count = render_frame_overlay(
                OUTPUT, frames[frame_id], scene, labels, color_menu.value, hold_id
            )
            with review_output:
                review_output.clear_output(wait=True)
                if hold_id is not None:
                    hold = holds_by_id[hold_id]
                    print(f"{labels[hold_id]} · {hold['id']} · {hold['hold_type']} · "
                          f"{hold['observation_count']} view(s)")
                    if review_3d:
                        print("3D matching status", hold["geometry_status"])
                        if hold.get("height_3d") is not None:
                            metric = hold["height_m"]
                            height = f"{metric:.3g} m" if metric is not None else f"{hold['height_3d']:.3g} reconstruction units"
                            print("Relative height", height)
                figure, axes = plt.subplots(1, 2, figsize=(13, 8))
                for axis, image, title in zip(axes, [original, overlay], ["Processed photo", f"{count} saved observations"]):
                    axis.imshow(image)
                    axis.set_title(title)
                    axis.axis("off")
                figure.tight_layout()
                display(figure)
                plt.close(figure)

        def choose_evidence_frames(change):
            hold_id = change["new"]
            if hold_id is None:
                frame_menu.options = frame_options
            else:
                evidence = {observation_frame[obs] for obs in holds_by_id[hold_id]["observation_ids"]}
                frame_menu.options = [option for option in frame_options if option[1] in evidence]
            show_frame()

        hold_menu.observe(choose_evidence_frames, names="value")
        frame_menu.observe(show_frame, names="value")
        color_menu.observe(show_frame, names="value")
        display(hold_menu, frame_menu, color_menu, review_output)
        show_frame()

## 7. Optional matching to annotated GLB

For blue V5 / yellow V3, use an explicit 4×4 transform from our reconstruction units to the annotated GLB's meter frame. Independent COLMAP and Polycam captures do not align automatically. The matcher assigns named hold objects one-to-one within a distance gate and retains unmatched holds.

In [ ]:
MATCH_ANNOTATED = False
ANNOTATED_GLB = WALL_C / "blue_v5" / "wallC_blueV5_marked.glb"
SCENE_TO_ANNOTATED = PROJECT / "data" / "scene_to_annotated.json"
if MATCH_ANNOTATED:
    if scene is None or scene["analysis_mode"] != "3d":
        raise ValueError("Export a 3D scene before matching it to the annotated GLB.")
    run_holds("match-annotated-glb", SCENE_PATH, "--annotated-glb", ANNOTATED_GLB,
              "--transform", SCENE_TO_ANNOTATED, "--output", OUTPUT / "scene_matched.json")
    matched_scene = json.loads((OUTPUT / "scene_matched.json").read_text())
    display(pd.DataFrame([
        {"hold": labels[h["id"]], **h["ground_truth_match"]}
        for h in matched_scene["holds"] if h["ground_truth_match"]
    ]))

## The same inference workflow on HPC

The implementation also runs without Jupyter. Configure the environment and ready checkpoint paths, then submit the analysis chain:

```bash
export DEMO_PROJECT=/scratch/NetID/hold_detection
export DEMO_PYTHON=/scratch/NetID/runtime/bin/python
export DEMO_SOURCE=/scratch/NetID/captures/wall.mov
export DEMO_RUN=/scratch/NetID/hold-runs/wallC
export DEMO_MODE=3d
export DEMO_SAM_WEIGHTS=/scratch/NetID/models/sam3/sam3.pt
export DEMO_TYPE_WEIGHTS=/scratch/NetID/models/hold_types_best.pt
export DEMO_TYPE_IMGSZ=640
export DEMO_COLMAP=/path/to/cuda-colmap/bin/colmap
bash scripts/slurm/submit_wall.sh
```

The chain is prepare → reconstruct → segment → export, with `afterok` dependencies. For a single photo use `DEMO_MODE=2d`; reconstruction is skipped. The alternative single-job entry point is `DEMO_STAGE=analyze sbatch scripts/slurm/wall-gpu.sbatch`.

These are text examples and are not submitted automatically. Model fitting remains in separate Python/SLURM jobs outside this notebook.

Review missing/extra masks, ambiguous types, registered-view count, repeated/unmapped holds, units and spatial estimates before giving the scene to the graph algorithm. Synthetic tests do not establish real wall accuracy. See [the complete pipeline guide](../docs/wall-pipeline.md).